# conduyt-pilot fine-tune (Phase 2)

SFT fine-tune of `unsloth/Qwen2.5-Coder-1.5B-Instruct` with QLoRA + Unsloth on a Kaggle T4 x2 instance, then merge + export to GGUF (Q4_K_M / Q5_K_M / Q8_0). MLC conversion lives in `convert_mlc.ipynb` (different env).

- Base model: https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct
- Dataset: private Kaggle dataset `conduyt-pilot-data` (mounted at `/kaggle/input/conduyt-pilot-data/`)
- Output adapter: `virgilvox/conduyt-pilot-1.5b-lora`
- Output GGUF: `virgilvox/conduyt-pilot-1.5b-gguf` (replace `virgilvox` with your HF user)

Wall-clock budget: roughly 30 to 60 minutes for ~150 examples x 3 epochs on a single T4.


In [ ]:
# Pinned versions. Unsloth's API moves; pinning keeps this notebook reproducible.
# Source for these pins: Unsloth's own current Kaggle Qwen2.5-Coder reference notebook,
# cross-checked against unsloth==2026.4.8 requires_dist on PyPI.
#
# Notes:
#  - Kaggle T4 x2 image ships with CUDA 12.x, torch >= 2.4. We let unsloth resolve
#    its own torch/xformers since the installer detects the runtime.
#  - `trl` is installed --no-deps to avoid pulling a transformers version that
#    conflicts with the unsloth pin.
#  - `datasets<4.4.0` is required by unsloth 2026.4.8.
#  - huggingface_hub is intentionally NOT pinned: transformers 4.56.2 caps it at
#    <1.0, while unsloth caps it at >=0.34.0. Let pip resolve.
#  - The fsspec / s3fs / gcsfs / bigframes warnings from Kaggle's pre-installed
#    packages are unrelated and safe to ignore for training.
#
!pip install -q --upgrade pip
!pip install -q unsloth==2026.4.8
!pip install -q --no-deps trl==0.22.2
!pip install -q transformers==4.56.2 \
                  peft==0.19.1 \
                  bitsandbytes==0.49.2 \
                  accelerate==1.13.0 \
                  datasets==4.3.0


In [ ]:
# Secrets. Kaggle's UserSecretsClient is the only way to access them at runtime.
# Required: HF_TOKEN (write scope) for pushing the adapter and GGUF.
# Optional: WANDB_API_KEY for run logging.
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")

try:
    os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
    USE_WANDB = True
except Exception:
    USE_WANDB = False
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))
print("WANDB on:    ", USE_WANDB)


In [ ]:
# Load the dataset. The Kaggle Dataset is added as input via the right-hand panel
# under the canonical slug `conduyt-pilot-data`; that mounts read-only at
# /kaggle/input/conduyt-pilot-data/.
import json
from datasets import Dataset

DATA_DIR = "/kaggle/input/conduyt-pilot-data"

def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_rows = load_jsonl(f"{DATA_DIR}/train.jsonl")
eval_rows  = load_jsonl(f"{DATA_DIR}/eval.jsonl")

train_ds = Dataset.from_list(train_rows)
eval_ds  = Dataset.from_list(eval_rows)

print(f"train: {len(train_ds)}  eval: {len(eval_ds)}")
print("--- first train example ---")
print(json.dumps(train_rows[0], indent=2)[:800])


In [ ]:
# Load the base model with 4-bit QLoRA via Unsloth's FastLanguageModel.
# dtype=None lets Unsloth pick the best for the T4 (which is fp16-only; bf16 is
# unsupported on T4 silicon).
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 4096
MODEL_NAME     = "unsloth/Qwen2.5-Coder-1.5B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LENGTH,
    load_in_4bit    = True,
    dtype           = None,
)
print(model)


In [ ]:
# LoRA config. r=16 is a reasonable middle ground for 1.5B; r=32 if you have
# more data later. target_modules covers all attention + MLP projections.
model = FastLanguageModel.get_peft_model(
    model,
    r                          = 16,
    lora_alpha                 = 32,
    target_modules             = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout               = 0.05,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 3407,
)


In [ ]:
# Format with the Qwen2.5 chat template. Each example already has a `messages`
# field; apply_chat_template renders it into the model's expected token sequence.
def format_example(row):
    text = tokenizer.apply_chat_template(
        row["messages"], tokenize=False, add_generation_prompt=False,
    )
    return { "text": text }

train_ds = train_ds.map(format_example, remove_columns=train_ds.column_names)
eval_ds  = eval_ds.map( format_example, remove_columns=eval_ds.column_names)

print("--- formatted example (truncated) ---")
print(train_ds[0]["text"][:1200])


In [ ]:
# SFT trainer config. fp16=True is required on T4. `num_train_epochs` scales with
# dataset size, which is what we want here (a fixed `max_steps` would underfit
# bigger datasets later).
from trl import SFTTrainer, SFTConfig

config = SFTConfig(
    output_dir                  = "/kaggle/working/outputs",
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps                = 10,
    num_train_epochs            = 3,
    learning_rate               = 2e-4,
    fp16                        = True,
    bf16                        = False,
    logging_steps               = 5,
    optim                       = "adamw_8bit",
    weight_decay                = 0.01,
    lr_scheduler_type           = "linear",
    seed                        = 3407,
    save_strategy               = "epoch",
    eval_strategy               = "epoch",
    report_to                   = "wandb" if USE_WANDB else "none",
    max_seq_length              = MAX_SEQ_LENGTH,
    dataset_text_field          = "text",
    packing                     = False,
)

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_ds,
    eval_dataset  = eval_ds,
    args          = config,
)


In [ ]:
import time
t0 = time.time()
stats = trainer.train()
elapsed = time.time() - t0
print(f"trained in {elapsed/60:.1f} min")
metrics = trainer.evaluate()
print("final eval:", metrics)


In [ ]:
# Save adapter locally and push to HF Hub.
ADAPTER_DIR = "/kaggle/working/adapter"
ADAPTER_REPO = "virgilvox/conduyt-pilot-1.5b-lora"

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

model.push_to_hub(ADAPTER_REPO, token=os.environ["HF_TOKEN"])
tokenizer.push_to_hub(ADAPTER_REPO, token=os.environ["HF_TOKEN"])
print("adapter pushed to", ADAPTER_REPO)


In [ ]:
# Quick sanity inference. Five hardcoded prompts that span the dataset's
# distribution. Eyeball the outputs; the goal at this stage is not perfect
# code but recognizable Moheeb voice + plausible structure.
FastLanguageModel.for_inference(model)

PROBES = [
    {"role": "user", "content": "Blink an LED on the ESP32-S3 DevKitC-1 onboard NeoPixel at 1 Hz."},
    {"role": "user", "content": "Read a BME280 over I2C and print T, P, H every 2 seconds. Target board: Adafruit Feather nRF52840 Sense."},
    {"role": "user", "content": "Write a platformio.ini for an Adafruit Feather nRF52840 Sense project that uses Adafruit_LSM6DS and Adafruit_BMP280."},
    {"role": "user", "content": "Conduyt firmware sketch on a Pico that exposes a writable angle datastream driving a servo on GP15."},
    {"role": "user", "content": "On an ESP32-S3, connect to WiFi and POST the millis() value as JSON to https://example.com/log every 30 seconds."},
]

for i, probe in enumerate(PROBES, 1):
    inputs = tokenizer.apply_chat_template(
        [probe], tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=400, do_sample=False)
    text = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"--- probe {i} ---")
    print(probe["content"])
    print("---")
    print(text)
    print()


In [ ]:
# Merge LoRA into the base weights, save full-precision merged model.
# Required for GGUF + MLC conversion downstream.
MERGED_DIR = "/kaggle/working/merged"

model.save_pretrained_merged(
    MERGED_DIR,
    tokenizer,
    save_method = "merged_16bit",
)
print("merged model at", MERGED_DIR)


In [ ]:
# GGUF export at three quants. save_pretrained_gguf() vendors llama.cpp's
# convert + quantize binaries; expect a few minutes per quant.
GGUF_REPO = "virgilvox/conduyt-pilot-1.5b-gguf"

for quant in ["q4_k_m", "q5_k_m", "q8_0"]:
    out_dir = f"/kaggle/working/gguf_{quant}"
    model.save_pretrained_gguf(
        out_dir,
        tokenizer,
        quantization_method = quant,
    )
    print(f"built {quant} -> {out_dir}")

# Push all three to one HF repo. We upload the .gguf files individually so the
# repo ends up with a flat layout (good for llama.cpp / Ollama / llamafile).
from huggingface_hub import HfApi, create_repo
import glob

api = HfApi(token=os.environ["HF_TOKEN"])
create_repo(GGUF_REPO, exist_ok=True, token=os.environ["HF_TOKEN"])

for path in sorted(glob.glob("/kaggle/working/gguf_*/**/*.gguf", recursive=True)):
    fname = path.split("/")[-1]
    api.upload_file(
        path_or_fileobj = path,
        path_in_repo    = fname,
        repo_id         = GGUF_REPO,
        repo_type       = "model",
    )
    print(f"pushed {fname}")

print("done. browse at https://huggingface.co/" + GGUF_REPO)


## Next step: MLC conversion

MLC LLM uses a different toolchain (TVM-based) and pulls in different CUDA pins, so we keep it in a separate notebook to avoid env conflicts. Open `convert_mlc.ipynb` next; it pulls the merged model from this run (either from `/kaggle/working/merged` if you're in the same session, or by re-uploading the merged folder to HF first).
